# 01 · Bronze: ingestão do `Chamados_Hora.CSV`

Como parte da camada de ingestão da arquitetura medalhão exigida no case, esta etapa resolve o desafio de integrar as bases heterogêneas da V-Credit - que chegam de sistemas diferentes com codificações, formatos de data e nomenclaturas distintos. Especificamente, ela lê o arquivo Chamados_Hora.CSV, que detalha o histórico de atendimento dos chamados contendo ID_Chamado, ID_Cliente e os respectivos horários de abertura, início e finalização, e grava esses dados diretamente em uma tabela Delta na camada bronze (workspace.bronze.chamados_hora) sem alterar nenhum valor, garantindo a fidelidade e a preservação total da base original antes das etapas de transformação.

**Decisões principais**

| Decisão | Motivo |
|---|---|
| Todas as colunas como texto (`string`) | Nenhum valor se perde por erro de conversão. A tipagem fica para a Silver. |
| Encoding e delimitador detectados | O arquivo está em Windows-1252. Ler como UTF-8 estragaria os acentos (como o "às" das datas). |
| Metadados em cada linha | `_source_file`, `_source_modified_at`, `_ingested_at` e `_run_id` permitem saber de onde veio cada linha e quando entrou. |
| Carga com `MERGE` por `_bronze_key` | Reexecutar sobre o mesmo CSV não duplica nada. A tabela só recebe inserções, então o histórico é mantido. |
| Conferência de contagem | Compara linhas do arquivo × linhas lidas pelo Spark × linhas inseridas, e registra o resultado em `ops.pipeline_run_log`. |

## Parâmetros
Catálogo, schemas, nome do arquivo e as 5 colunas que o CSV deve ter. O fuso da sessão é Brasília, porque o arquivo não traz fuso e os horários são locais.

In [0]:
import os, re, uuid, codecs
from datetime import datetime
from pyspark.sql import functions as F, Window, DataFrame

CATALOG        = "workspace"
SCHEMA_BRONZE  = "bronze"
SCHEMA_SILVER  = "silver"
SCHEMA_GOLD    = "gold"
SCHEMA_OPS     = "ops"            # log de execução / observabilidade
VOLUME_LANDING = "landing"        # onde o CSV é depositado (Unity Catalog Volume)
VOLUME_EXPORT  = "export"         # CSVs da camada Gold para o Power BI (usado no notebook Gold)
ARQUIVO_ORIGEM = "Chamados_Hora.CSV"   # a busca ignora maiúsculas/minúsculas

T_BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}.chamados_hora"
T_LOG    = f"{CATALOG}.{SCHEMA_OPS}.pipeline_run_log"

# Layout esperado do arquivo (nomes já normalizados: minúsculo, sem acento)
COLS_ESPERADAS = ["id_chamado", "id_cliente", "hora_abertura_chamado",
                  "hora_inicio_atendimento", "hora_finalizacao_atendimento"]

try:  # horário de Brasília: o CSV não traz fuso e os horários são locais
    spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
except Exception:
    pass

## 0. Ambiente
Cria os schemas (`bronze`, `silver`, `gold`, `ops`), os volumes (`landing` para o CSV, `export` para a Gold) e a tabela de log.

In [0]:
for schema in (SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OPS):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for vol in (VOLUME_LANDING, VOLUME_EXPORT):
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}.{vol}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_LOG} (
  run_id STRING, executado_em TIMESTAMP, etapa STRING, tabela STRING,
  linhas_lidas BIGINT, linhas_gravadas BIGINT, status STRING, detalhe STRING, fingerprint STRING
) COMMENT 'Log de execução do pipeline (auditoria e prova de idempotência)'
""")
print("Ambiente pronto: schemas bronze/silver/gold/ops, volumes landing/export e tabela de log.")

Ambiente pronto: schemas bronze/silver/gold/ops, volumes landing/export e tabela de log.


## Funções auxiliares
Localizar o arquivo no volume, detectar encoding e delimitador, contar as linhas do arquivo, normalizar os nomes das colunas (minúsculo, sem acento) e registrar cada execução no log.

In [0]:
def localizar_arquivo(diretorio: str, nome: str) -> str:
    """Procura o arquivo ignorando maiúsculas/minúsculas (Volumes diferenciam caixa)."""
    if not os.path.isdir(diretorio):
        raise FileNotFoundError(f"Diretório não encontrado: {diretorio}. Rode a seção '0. Ambiente' deste notebook.")
    for f in os.listdir(diretorio):
        if f.lower() == nome.lower():
            return os.path.join(diretorio, f)
    raise FileNotFoundError(f"'{nome}' não está em {diretorio}. Arquivos presentes: {os.listdir(diretorio)}. "
                            "Envie o CSV em Catalog > workspace > bronze > Volumes > landing > 'Upload to this volume' e execute novamente.")


def detectar_formato_csv(caminho: str, amostra_bytes: int = 50_000_000) -> dict:
    """Detecta encoding e delimitador (as bases vêm de sistemas diferentes, conforme o case)."""
    with open(caminho, "rb") as fh:
        b = fh.read(amostra_bytes)
    if b[:2] in (b"\xff\xfe", b"\xfe\xff"):
        enc_spark, enc_py = "UTF-16", "utf-16"
    else:
        try:  # decoder incremental tolera um caractere multibyte cortado no fim da amostra
            codecs.getincrementaldecoder("utf-8")().decode(b, final=False)
            enc_spark, enc_py = "UTF-8", "utf-8-sig"
        except UnicodeDecodeError:
            enc_spark, enc_py = "windows-1252", "cp1252"   # é o caso do Chamados_Hora.CSV
    primeira = b.decode(enc_py, errors="replace").splitlines()[0]
    sep = max([",", ";", "\t", "|"], key=primeira.count)
    return {"encoding_spark": enc_spark, "encoding_py": enc_py, "sep": sep}


def contar_linhas_dados(caminho: str) -> int:
    """Linhas de dados do arquivo (sem cabeçalho), para reconciliar com o que o Spark leu."""
    with open(caminho, "rb") as fh:
        return sum(1 for _ in fh) - 1


def normalizar_nome_coluna(c: str) -> str:
    import unicodedata
    c = c.replace("\ufeff", "").strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()


def registrar_log(etapa: str, tabela: str, linhas_lidas=None, linhas_gravadas=None,
                  status: str = "OK", detalhe: str = "", fingerprint=None):
    """Trilha de auditoria de cada execução (append: é log, não dado de negócio)."""
    row = [(RUN_ID, datetime.now(), etapa, tabela, linhas_lidas, linhas_gravadas, status, detalhe, fingerprint)]
    schema = ("run_id string, executado_em timestamp, etapa string, tabela string, linhas_lidas long, "
              "linhas_gravadas long, status string, detalhe string, fingerprint string")
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(T_LOG)


RUN_ID = uuid.uuid4().hex[:12]

## Preparação da Bronze (tudo texto + rastreabilidade)

**Idempotência na Bronze:** cada linha recebe uma chave determinística `_bronze_key = sha256(hash do conteúdo + nº da ocorrência)`.
O `MERGE ... WHEN NOT MATCHED THEN INSERT` só insere o que ainda não existe. Reexecutar sobre o mesmo CSV não insere nada.
O "nº da ocorrência" preserva linhas realmente repetidas no arquivo (duas linhas idênticas = duas chaves distintas), em vez de escondê-las.

In [0]:
def preparar_bronze(df_raw: DataFrame, nome_arquivo: str, modificado_em) -> DataFrame:
    """Normaliza cabeçalhos, valida o contrato de colunas e acrescenta metadados técnicos. Nenhum valor é alterado."""
    d = df_raw.toDF(*[normalizar_nome_coluna(c) for c in df_raw.columns])
    faltantes = [c for c in COLS_ESPERADAS if c not in d.columns]
    if faltantes:
        raise ValueError(f"Contrato de colunas violado. Faltam: {faltantes}. Encontradas: {d.columns}")
    d = d.select(*[F.col(c).cast("string") for c in d.columns])       # Bronze = tudo string

    conteudo = F.concat_ws("\u001f", *[F.coalesce(F.col(c), F.lit("\u0000NULL")) for c in d.columns])
    d = d.withColumn("_row_hash", F.sha2(conteudo, 256))
    seq = Window.partitionBy("_row_hash").orderBy(F.lit(1))
    d = d.withColumn("_dup_seq", F.row_number().over(seq))
    return (d.withColumn("_bronze_key", F.sha2(F.concat_ws("|", F.col("_row_hash"), F.col("_dup_seq")), 256))
             .withColumn("_source_file", F.lit(nome_arquivo))
             .withColumn("_source_modified_at", F.lit(modificado_em).cast("timestamp"))
             .withColumn("_ingested_at", F.current_timestamp())
             .withColumn("_run_id", F.lit(RUN_ID))
             .drop("_row_hash", "_dup_seq")
             .select(*COLS_ESPERADAS, *[c for c in d.columns if c not in COLS_ESPERADAS + ["_row_hash", "_dup_seq"]],
                     "_bronze_key", "_source_file", "_source_modified_at", "_ingested_at", "_run_id"))

## 1. Localizar o arquivo e detectar o formato
Encontra o CSV no volume e identifica encoding e separador antes de ler, além de contar as linhas de dados para a conferência posterior.

In [0]:
pasta = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/{VOLUME_LANDING}"
caminho = localizar_arquivo(pasta, ARQUIVO_ORIGEM)
nome_arquivo = os.path.basename(caminho)
modificado_em = datetime.fromtimestamp(os.stat(caminho).st_mtime)

fmt = detectar_formato_csv(caminho)
linhas_arquivo = contar_linhas_dados(caminho)
print(f"Arquivo: {caminho}\nFormato detectado: {fmt}\nLinhas de dados no arquivo: {linhas_arquivo:,}")

Arquivo: /Volumes/workspace/bronze/landing/Chamados_Hora.CSV
Formato detectado: {'encoding_spark': 'windows-1252', 'encoding_py': 'cp1252', 'sep': ','}
Linhas de dados no arquivo: 131,838


## 2. Ler como texto e preparar a Bronze
Leitura sem inferência de tipos. Depois da leitura, o notebook confere se a quantidade de linhas bate com a do arquivo. Se não bater, o status da execução fica como `ALERTA`.

In [0]:
raw = (spark.read.format("csv")
       .option("header", True)
       .option("sep", fmt["sep"])
       .option("encoding", fmt["encoding_spark"])
       .option("inferSchema", False)       # tudo texto
       .load(caminho))

df_b = preparar_bronze(raw, nome_arquivo, modificado_em)
linhas_lidas = df_b.count()
print(f"Linhas lidas pelo Spark: {linhas_lidas:,}")
status = "OK" if linhas_lidas == linhas_arquivo else "ALERTA"
if status == "ALERTA":
    print(f"⚠ Divergência: arquivo={linhas_arquivo:,} × Spark={linhas_lidas:,} (campos com quebra de linha ou linhas corrompidas?)")

Linhas lidas pelo Spark: 131,838


## 3. Carga idempotente
O `MERGE` só insere as linhas cuja `_bronze_key` ainda não existe na tabela. O resultado (lidas, inseridas, já existentes) é impresso e gravado no log.

In [0]:
try:  # permite que novas colunas no CSV entrem sem quebrar (schema evolution)
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
except Exception:
    pass

if not spark.catalog.tableExists(T_BRONZE):
    df_b.limit(0).write.format("delta").saveAsTable(T_BRONZE)

df_b.createOrReplaceTempView("v_bronze_novo")
antes = spark.table(T_BRONZE).count()

spark.sql(f"""
MERGE INTO {T_BRONZE} AS t
USING v_bronze_novo AS s
ON t._bronze_key = s._bronze_key
WHEN NOT MATCHED THEN INSERT *
""")

depois = spark.table(T_BRONZE).count()
inseridas = depois - antes
spark.sql(f"COMMENT ON TABLE {T_BRONZE} IS 'Bronze: copia fiel do Chamados_Hora.CSV (texto) com metadados de ingestao. Append-only.'")

print(f"Antes: {antes:,} | Inseridas agora: {inseridas:,} | Já existiam (ignoradas): {linhas_lidas - inseridas:,} | Total Bronze: {depois:,}")
registrar_log("bronze", T_BRONZE, linhas_lidas, inseridas, status,
              f"arquivo={nome_arquivo}; encoding={fmt['encoding_spark']}; sep={fmt['sep']!r}; linhas_arquivo={linhas_arquivo}")

Antes: 131,838 | Inseridas agora: 0 | Já existiam (ignoradas): 131,838 | Total Bronze: 131,838


## 4. Conferência visual

In [0]:
display(spark.table(T_BRONZE).orderBy("_ingested_at").limit(10))

id_chamado,id_cliente,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,_bronze_key,_source_file,_source_modified_at,_ingested_at,_run_id
118623,54800735508,14/06/2026 às 16:54:10,14/06/2026 às 17:11:15,14/06/2026 às 17:57:11,2e4cd7c0f9087d02dd53400047c68c472b91023b8fb5ea7bbec56da76da546b8,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
65899,19834956312,29/12/2025 às 14:16:07,29/12/2025 às 14:24:30,29/12/2025 às 14:42:09,e81df47a830ef371fd99348bb7c4237bbf0908e585ebfe4db3b924207dc06383,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
27940,56723902217,05/06/2025 às 18:17:25,null,05/06/2025 às 18:20:08,25709bf761d23ad949dae9a5efbc57a9c7d283e59e9a947a86c3b3d5e67e4fee,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
18540,93467645740,13/04/2025 às 18:36:31,13/04/2025 às 18:36:31,13/04/2025 às 18:38:14,431cf8d9241129ac5ad99b0be468a188dfe39fae80912a8416c14b3920a84c93,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
66521,68778585875,27/12/2025 às 12:03:34,27/12/2025 às 12:09:26,27/12/2025 às 12:26:43,dcacea1677cac7176610c650d2db66c0d600ac6fba21e081e8cae901f388a629,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
59506,11929310165,03/11/2025 às 19:13:35,03/11/2025 às 19:27:51,03/11/2025 às 19:34:21,d7e0ff5a35fea97c62932fefd6d364d22ee4e74de01fc2c8b471e8070f91de85,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
34381,25316091825,07/07/2025 às 13:31:56,07/07/2025 às 13:39:18,07/07/2025 às 14:01:20,264a51df80c615b7308ba9487ffd74a395a8e2d24c5c716aab70b3969dec1965,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
100625,89154442587,24/04/2026 às 19:58:58,24/04/2026 às 20:04:18,24/04/2026 às 20:26:51,1c332042e442661e8017e8c67c40042c19462df41a684dc8a9d8b15544bb6ca3,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
6448,67658336742,14/02/2025 às 10:44:36,14/02/2025 às 10:46:39,14/02/2025 às 10:50:10,a08ec1c41c82d7a470d5d8d463230813ab022c3ba0281dfaef780178d90e34c0,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
34436,86390723640,14/07/2025 às 17:41:20,14/07/2025 às 17:41:20,14/07/2025 às 17:44:00,cd4a8ccb854c22b57af66e9cf951b14834e8a7be3a380c94211f1c20d8c4e96f,Chamados_Hora.CSV,2026-10-26T09:37:33.000-03:00,2026-10-26T09:38:08.099-03:00,30899ba1f994
